# Build Rag pipeline

### Load & Split document

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. create a sample documnet
content = """
LangChain is a framework for developing applications powered by language models. It enables applications that are context-aware and reason across problems. Key components of LangChain include Models, Prompts, Chains, and Memory. Chains allow you to combine multiple components together.

RAG (Retrieval-Augmented Generation) is a popular technique used in LangChain.
"""

with open("langchain_intro.txt", "w", encoding="utf-8") as f:
    f.write(content)

# 2. Load the document
loader = TextLoader("langchain_intro.txt", encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")

# 3. Split document (chunking)

text_splitter = RecursiveCharacterTextSplitter(
chunk_size=100, chunk_overlap=20,
length_function=len,
separators=["\n\n", "\n", "", ""]
)

splits = text_splitter.split_documents (documents)


print("Split into {len(splits)} chunks.") 
for i, split in enumerate(splits):
    print(f"Chunk {i}: {split.page_content}")


C:\Users\HASNAIN\AppData\Local\Temp\ipykernel_1260\36859300.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\HASNAIN\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 1 document(s).
Split into {len(splits)} chunks.
Chunk 0: LangChain is a framework for developing applications powered by language models. It enables applica
Chunk 1: . It enables applications that are context-aware and reason across problems. Key components of LangC
Chunk 2: components of LangChain include Models, Prompts, Chains, and Memory. Chains allow you to combine mu
Chunk 3: ow you to combine multiple components together.
Chunk 4: RAG (Retrieval-Augmented Generation) is a popular technique used in LangChain.


### Generate embeddings

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Creating embeddings and indexing in FAISS...")

db = FAISS.from_documents(
    documents=splits,
    embedding=embedding_function
)

print("Vector Database created successfully!")
print(f"Stored {len(splits)} vectors.")

C:\Users\HASNAIN\AppData\Local\Temp\ipykernel_1260\3513691142.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 292.05it/s]


Creating embeddings and indexing in FAISS...
Vector Database created successfully!
Stored 5 vectors.


## Load the LLM

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

#---- Configuration----
model_id = "Qwen/Qwen2-0.5B-Instruct"

# 1. Load model & Tokenizer
print(f"Loading model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

# 2. create HF pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.1,
    do_sample=True
)

# 3. Langchain LLM Wrapper
raw_llm = HuggingFacePipeline(pipeline=pipe)


# 4. Define the Qwen chat formatter
def format_for_qwen(input_dict):
    messages = [
        {
            "role": "system",
            "content": """
You are a helpful AI assistant that answers questions strictly based on the provided context.
Do not use any external knowledge or make up information.
If the answer is not in the context, respond exactly with: "I don't know based on the provided context."
Always start your response with "Answer:" followed by the answer or the "I don't know" statement.
"""
        },
        {
            "role": "user",
            "content": f"""
Context:
{input_dict['context']}

Question:
{input_dict['question']}
"""
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

# 5. Generation Function
def generate_with_qwen(formatted_prompt):
    response = raw_llm.invoke(formatted_prompt)
    generated = response.split(formatted_prompt)[-1].strip()

    if generated.startswith("Answer:"):
        generated = generated.split("Answer:", 1)[-1].strip()

    return generated

# 6. Create LLM chain with formatting
llm_with_format = (
    RunnableLambda(format_for_qwen)
    | RunnableLambda(generate_with_qwen)
    | StrOutputParser()
)

print("LLM setup complete!")

Loading model: Qwen/Qwen2-0.5B-Instruct...


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 261.12it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM setup complete!


## Create the pipeline

In [4]:
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

retriever = db.as_retriever(search_kwargs={"k": 2})

rag_chain = {
    "context": retriever,
    "question": RunnablePassthrough()
} | RunnablePassthrough.assign(
    answer=lambda x: llm_with_format.invoke(
        {
            "context": "\n\n".join([doc.page_content for doc in x["context"]]),
            "question": x["question"]
        }
    )
)

# Create Flask APP

##### Note: This app simply call the endpoint RAG

### Import the Libraries

In [5]:
%pip install flask flask-cors

Note: you may need to restart the kernel to use updated packages.


In [6]:
import threading
import time
import requests
from flask import Flask, request, jsonify
from flask_cors import CORS
import logging

### 1. Helper Functions


In [7]:
def create_qwen_prompt(user_text, history=None):
    if history is None:
        history = []

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."}
    ]

    messages.extend(history)
    messages.append({"role": "user", "content": user_text})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def clean_qwen_response(response: str, prompt: str) -> str:
    if prompt in response:
        return response.replace(prompt, "").strip()

### 2. Flask APP Setup

In [8]:
app = Flask(__name__)
CORS(app)    #Cross origin resource sharing
log = logging.getLogger('werkzeug') 
log.setLevel(logging.ERROR)

### 3. Routes

In [9]:
@app.route('/rag_query', methods=['POST'])
def rag_query():
    if 'rag_chain' not in globals():
        return jsonify({"error": "RAG chain not initialized"}), 501

    try:
        data = request.json
        response = rag_chain.invoke(data.get('question', ''))
        clean_answer = clean_qwen_response(response['answer'], "Answer: ")
        sources = [doc.page_content for doc in response['context']]
        return jsonify({"answer": clean_answer, "sources": sources})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

### 4. Start Server

In [10]:
def run_flask():
    print("Flask Server started on http://127.0.0.1:5000")
    app.run(port=5000, use_reloader=False)

t = threading.Thread(target=run_flask)
t.daemon = True
t.start()

print("   \nWaiting for server to be ready...")
time.sleep(3)

Flask Server started on http://127.0.0.1:5000   
Waiting for server to be ready...

 * Serving Flask app '__main__'
 * Debug mode: off


### Test the RAG

In [11]:
print("\n[TEST] Endpoint: /rag_query")
print("-" * 40)

try:
    resp = requests.post(
        "http://127.0.0.1:5000/rag_query",
        json={"question": "what does RAG stand for?"}
    )

    if resp.status_code == 200:
        print("User: What does RAG stand for?")
        print(f"Answer: {resp.json()['answer']}")
        print("Context Used:")

        for source in resp.json()['sources']:
            print(f" - {source}")
    else:
        print(f"Error: {resp.status_code}")

except Exception as e:
    print(f"Failed: {e}")

time.sleep(1)


[TEST] Endpoint: /rag_query
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


User: What does RAG stand for?
Answer: None
Context Used:
 - RAG (Retrieval-Augmented Generation) is a popular technique used in LangChain.
 - . It enables applications that are context-aware and reason across problems. Key components of LangC
